In [2]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

## Process the dataset by using the positive impressions and concatenating them with the history, and splitting the history of each user using the 80:20 ratio

In [42]:
def combine_history_and_impressions(row):
    """
    Parses positive impressions and concatenates them with existing history.
    """
    # Handle cases where history is missing (NaN)
    history = row['history'] if pd.notna(row['history']) else ''

    # Split the impressions string and find clicked articles (ending in '-1')
    # Then, strip the '-1' to get the pure article ID
    positive_impressions = [
        imp.split('-')[0] for imp in row['impressions'].split(' ') if imp.endswith('-1')
    ]

    # If there are no positive impressions, just return the original history
    if not positive_impressions:
        return history

    # Join the found positive impression IDs into a space-separated string
    positive_impressions_str = ' '.join(positive_impressions)
    
    # Combine history and the new positive impressions
    # This logic avoids adding an unnecessary space if the history is empty
    if history:
        return history + ' ' + positive_impressions_str
    else:
        return positive_impressions_str
    

# Isolate the positive clicks from each impression log
def get_positive_clicks(impressions_str):
    """Extracts only the IDs of clicked articles (ending in '-1')"""
    if pd.isna(impressions_str):
        return ''
    
    positive_clicks = [
        imp.split('-')[0] for imp in impressions_str.split(' ') if imp.endswith('-1')
    ]
    return ' '.join(positive_clicks)

# Group by user and their initial history, then aggregate the positive clicks
def aggregate_clicks(series):
    """Joins a series of strings, filtering out any empty ones."""
    return ' '.join(item for item in series if item)


# Combine the original history with the aggregated positive clicks
def create_full_history(row):
    """Concatenates the original history with all aggregated clicks for that group."""
    history = row['history']
    new_clicks = row['positive_clicks']
    
    if history and new_clicks:
        return history + ' ' + new_clicks
    # If original history is empty, just return the new clicks.
    # If new clicks are empty, just return the history.
    else:
        return history + new_clicks


In [45]:
# --- Load and preprocess full dataset ---
behaviors_df = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

behaviors_df = behaviors_df[['user_id', 'history', 'impressions']]

# Handle cases where history might be missing (NaN)
behaviors_df['history'].fillna('', inplace=True)


behaviors_df['positive_clicks'] = behaviors_df['impressions'].apply(get_positive_clicks)

# The core of the solution: group by user/history and aggregate the new clicks.
# The .apply() will join the 'positive_clicks' strings from all rows in the group.
aggregated_df = behaviors_df.groupby(['user_id', 'history'])['positive_clicks'].apply(aggregate_clicks).reset_index()

aggregated_df['history'] = aggregated_df.apply(create_full_history, axis=1)

# 4. Final Cleanup: Select and rename columns to match the desired output
final_df = aggregated_df[['user_id', 'history']]

behaviors_df = final_df.copy()


# Parse history: string -> list
behaviors_df['history'] = behaviors_df['history'].fillna('').apply(str.split)

# --- Map user_id and news_id to integers ---
all_user_ids = sorted(behaviors_df['user_id'].unique())
user2int = {uid: idx for idx, uid in enumerate(all_user_ids)}
behaviors_df['user_id'] = behaviors_df['user_id'].map(user2int)

all_news_ids = sorted(set(nid for sublist in behaviors_df['history'] for nid in sublist))
news2int = {nid: idx+1 for idx, nid in enumerate(all_news_ids)}

print(f"Total unique users: {len(user2int)}")
print(f"Total unique news items: {len(news2int)}")

# Convert history to list of int IDs
behaviors_df['history'] = behaviors_df['history'].apply(
    lambda x: [news2int[n] for n in x if n in news2int]
)

# --- Train/Val/Test user split ---
unique_users = np.array(behaviors_df['user_id'].unique())
train_users, val_test_users = train_test_split(unique_users, test_size=0.2, random_state=42)
val_users, test_users = train_test_split(val_test_users, test_size=0.5, random_state=42)

train_users_set = set(train_users)
val_users_set = set(val_users)
test_users_set = set(test_users)

print(f"Train users: {len(train_users_set)}")
print(f"Validation users: {len(val_users_set)}")
print(f"Test users: {len(test_users_set)}")

# --- Create all records (full set, with correct format) ---
train_records, val_records, test_records = [], [], []

append_train = train_records.append
append_val = val_records.append
append_test = test_records.append

for row in behaviors_df.itertuples(index=False):
    uid, history = row.user_id, row.history
    hlen = len(history)

    if hlen < 2:
        continue

    if uid in train_users_set:
        train_records += [
            {
                'user_id': uid,
                'user_history': str(history[:i]),
                'next_item': str(history[i])
            }
            for i in range(1, hlen)
        ]

    elif uid in val_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_val({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

    elif uid in test_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_test({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

# --- Convert to DataFrames ---
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

# --- Show Summary ---
print("\n--- Training Data Sample (Lower Triangular Logic) ---")
print(f"Total training examples created: {len(train_df)}")
print(train_df.head())

print("\n--- Validation Data Sample (80/20 Split) ---")
print(f"Total validation examples created: {len(val_df)}")
print(val_df.head())

print("\n--- Test Data Sample (80/20 Split) ---")
print(f"Total test examples created: {len(test_df)}")
print(test_df.head())


Total unique users: 50000
Total unique news items: 39865
Train users: 40000
Validation users: 5000
Test users: 5000

--- Training Data Sample (Lower Triangular Logic) ---
Total training examples created: 887453
   user_id                        user_history next_item
0        0                              [6865]     16272
1        0                       [6865, 16272]     24442
2        0                [6865, 16272, 24442]     31060
3        0         [6865, 16272, 24442, 31060]     27254
4        0  [6865, 16272, 24442, 31060, 27254]      5987

--- Validation Data Sample (80/20 Split) ---
Total validation examples created: 4990
   user_id                                 given_user_history   
0        4  [9485, 13707, 24959, 15674, 39600, 18023, 2346...  \
1        6  [28618, 6754, 5759, 39737, 12825, 2228, 27671,...   
2       38  [24296, 4631, 17275, 5841, 19507, 38644, 24336...   
3       61  [28018, 21237, 27933, 4172, 11509, 7925, 10603...   
4       71  [17277, 2390, 16814, 210

In [46]:
import ast

def print_dataset_stats(name, df, history_col, predict_col=None):
    # Parse lists from string-formatted columns
    all_items = []
    for row in df.itertuples(index=False):
        history = ast.literal_eval(getattr(row, history_col))
        all_items.extend(history)
        if predict_col:
            predicted = ast.literal_eval(getattr(row, predict_col))
            all_items.extend(predicted)

    print(f"\n--- {name.upper()} SET STATS ---")
    print(f"Rows: {len(df)}")
    print(f"Unique users: {df['user_id'].nunique()}")
    print(f"Unique news items: {len(set(all_items))}")

# 📊 Print stats for all datasets
#print_dataset_stats("train", train_df, history_col='user_history')
print_dataset_stats("validation", val_df, history_col='given_user_history', predict_col='predicting_items')
print_dataset_stats("test", test_df, history_col='given_user_history', predict_col='predicting_items')



--- VALIDATION SET STATS ---
Rows: 4990
Unique users: 4990
Unique news items: 16692

--- TEST SET STATS ---
Rows: 4990
Unique users: 4990
Unique news items: 16497


In [ ]:
# --- (Optional) Save mappings for reference ---
pd.Series(user2int).to_csv("dataset/user2idx.tsv", sep="\t", header=False)
pd.Series(news2int).to_csv("dataset/news2idx.tsv", sep="\t", header=False)

# --- Save final datasets ---
train_df.to_csv('dataset/training.tsv', sep='\t', index=False)
val_df.to_csv('dataset/validation.tsv', sep='\t', index=False)
test_df.to_csv('dataset/test.tsv', sep='\t', index=False)

print("Saved: training.tsv, validation.tsv, test.tsv, and mappings")

Saved: training.tsv, validation.tsv, test.tsv, and mappings


In [ ]:
# Load the tsv files
train_df = pd.read_csv('dataset/training.tsv', sep='\t')
val_df = pd.read_csv('dataset/validation.tsv', sep='\t')
test_df = pd.read_csv('dataset/test.tsv', sep='\t')
# Display the first few rows of each dataframe
print("Training Data:")
print(train_df.head())
print("\nValidation Data:")
print(val_df.head())    
print("\nTest Data:")
print(test_df.head())
# Display the last few rows of each dataframe
print("\nLast few rows of Training Data:")
print(train_df.tail())
print("\nLast few rows of Validation Data:")
print(val_df.tail())
print("\nLast few rows of Test Data:")
print(test_df.tail())

Training Data:
   user_id                        user_history  next_item
0        0                              [6865]      16272
1        0                       [6865, 16272]      24442
2        0                [6865, 16272, 24442]      31060
3        0         [6865, 16272, 24442, 31060]      27254
4        0  [6865, 16272, 24442, 31060, 27254]       5987

Validation Data:
   user_id                                 given_user_history   
0        4  [9485, 13707, 24959, 15674, 39600, 18023, 2346...  \
1        6  [28618, 6754, 5759, 39737, 12825, 2228, 27671,...   
2       38  [24296, 4631, 17275, 5841, 19507, 38644, 24336...   
3       61  [28018, 21237, 27933, 4172, 11509, 7925, 10603...   
4       71  [17277, 2390, 16814, 21065, 38884, 23970, 1715...   

                                    predicting_items  
0                    [19318, 252, 9289, 32644, 3631]  
1                                [9596, 11376, 4060]  
2                          [17463, 24517, 308, 3063]  
3  [2179

In [50]:
uid = np.array([4, 6, 38])  # Example user IDs to filter
step_next_items = val_df.loc[val_df['user_id'].isin(uid), 'predicting_items'].values
print(step_next_items)

['[19318, 252, 9289, 32644, 3631]' '[9596, 11376, 4060]'
 '[17463, 24517, 308, 3063]']


In [51]:
import ast

def validate_val_df(val_df):
    print("\n[Validation Precheck]")

    invalid_rows = 0
    total_rows = len(val_df)

    for i, row in val_df.iterrows():
        try:
            # Parse the list string into actual list (if not already a list)
            predicting_items = row['predicting_items']
            if isinstance(predicting_items, str):
                predicting_items = ast.literal_eval(predicting_items)

            if not isinstance(predicting_items, list):
                raise ValueError(f"Invalid type at row {i}")

            if len(predicting_items) == 0:
                print(f"[Warning] Empty ground truth at row {i} (user_id={row['user_id']})")
                invalid_rows += 1

        except Exception as e:
            print(f"[Error] Failed to parse row {i}: {e}")
            invalid_rows += 1

    print(f"✅ Checked {total_rows} validation rows.")
    print(f"❌ Found {invalid_rows} rows with invalid or empty ground truths.\n")

    assert (total_rows - invalid_rows) == len(val_df), \
        "Mismatch: Some validation rows are malformed. Please fix before training."

validate_val_df(val_df)



[Validation Precheck]
✅ Checked 4990 validation rows.
❌ Found 0 rows with invalid or empty ground truths.

